# Étape 8 — Stratégies combinées (7 candidates)
## MASI Hybrid Forecasting System

On compare 7 stratégies sur la fenêtre TEST canonique (948 jours, 5 bps one-way) :

1. Buy & Hold (référence passive)
2. CNN-LSTM nu (référence étape 6)
3. CNN-LSTM + HMM-gate (skip Neutral)
4. CNN-LSTM + risk-gate (skip σ_GARCH high)
5. CNN-LSTM + HMM + risk-gate
6. CNN-LSTM × VaR-budget (inspiré repo `masi-risk-research-notebooks`)
7. CNN-LSTM × HMM-conditional budget (inspiré repo)

Validation : **Deflated Sharpe Ratio** (N=7) + **Diebold-Mariano** vs CNN-LSTM nu + décomposition régime-conditionnelle.

## 1. Exécution du script

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

import importlib.util as _ilu, sys as _sys; _spec = _ilu.spec_from_file_location('e8', '../scripts/08_strategies.py'); e8 = _ilu.module_from_spec(_spec); _sys.modules['e8'] = e8; _spec.loader.exec_module(e8)
out_df, full = e8.main()

## 2. Tableau récapitulatif (Sharpe / Sortino / MDD / Calmar / equity)

In [ ]:
rows = []
for k, m in full['strategies'].items():
    rows.append({
        'stratégie': m['label'],
        'mode': m['mode'],
        'Sharpe': round(m['sharpe'], 3),
        'Sortino': round(m['sortino'], 3),
        'MDD': f"{m['max_drawdown']*100:+.2f}%",
        'Calmar': round(m['calmar'], 2),
        'eq finale': round(m['final_equity'], 3),
        '% jours actifs': f"{m['pct_days_active']*100:.0f}%",
        'turnover moy.': round(m['turnover_mean'], 3),
        'n trades': m['n_trades'],
    })
pd.DataFrame(rows).set_index('stratégie')

## 3. Deflated Sharpe Ratio (N=7 trials)

In [ ]:
rows = []
for k, d in full['deflated_sharpe'].items():
    rows.append({
        'stratégie': full['strategies'][k]['label'],
        'SR daily': round(d['sr_daily'], 4),
        'SR0 seuil': round(d['sr0_threshold'], 4),
        'skew': round(d['skew'], 3),
        'kurt': round(d['kurt_pearson'], 2),
        'DSR vs SR0': round(d['deflated_sharpe_psr_vs_sr0'], 3),
        'PSR vs 0': round(d['psr_vs_zero'], 3),
        'verdict': d['verdict'],
    })
pd.DataFrame(rows).set_index('stratégie')

## 4. Diebold-Mariano vs CNN-LSTM nu (référence)

In [ ]:
rows = []
for k, d in full['diebold_mariano_vs_cnn_lstm_nu'].items():
    rows.append({
        'stratégie vs CNN-LSTM nu': full['strategies'][k]['label'],
        'mean(r_strat − r_nu)': round(d['mean_diff'], 5),
        'DM stat': round(d['dm_stat'], 2),
        'p-value': round(d['pvalue'], 3),
        'verdict': 'SIGNIFICATIF' if d['pvalue'] < 0.05 else 'non significatif',
    })
pd.DataFrame(rows).set_index('stratégie vs CNN-LSTM nu')

## 5. Décomposition régime-conditionnelle (Sharpe par régime HMM)

In [ ]:
rows = []
for k, m in full['strategies'].items():
    rc = m['regime_conditional']
    rows.append({
        'stratégie': m['label'],
        'Bear (n=168)': round(rc['Bear']['sharpe'], 2),
        'Neutral (n=371)': round(rc['Neutral']['sharpe'], 2),
        'Bull (n=409)': round(rc['Bull']['sharpe'], 2),
    })
pd.DataFrame(rows).set_index('stratégie')

## 6. Plots

In [ ]:
Image(os.path.join('../reports/figures/etape8', 'etape8_equity_curves.png'))

In [ ]:
Image(os.path.join('../reports/figures/etape8', 'etape8_drawdowns.png'))

In [ ]:
Image(os.path.join('../reports/figures/etape8', 'etape8_regime_heatmap.png'))

In [ ]:
Image(os.path.join('../reports/figures/etape8', 'etape8_sharpe_mdd_scatter.png'))

In [ ]:
Image(os.path.join('../reports/figures/etape8', 'etape8_dsr_summary.png'))

## 7. Conclusion (détaillée dans `outputs/etape8/report.md`)

- **Vainqueur clair : CNN-LSTM + HMM-gate** — Sharpe **+1.71**, Calmar **+2.84**, MDD **−6.00 %**, **DSR = 0.997 (PUBLISHABLE)**. Confirme la « future work » de l'étape 6.
- **Combiner HMM-gate + risk-gate dégrade** (Sharpe 1.06 vs 1.71 seul) : le risk-gate retire des jours Bull à haute vol qui sont profitables. **L'intersection est trop restrictive.**
- **VaR-budget (continu)** : Sharpe +1.22, MDD modeste (-10.6 %), mais turnover ~3× supérieur (588-611 trades vs 180-221). Trade-off : meilleur risk-adjusted que nu, moins bon que gating, plus coûteux en pratique.
- **Buy & Hold** est dominé sur tous les axes : Sharpe +0.88, MDD −20.7 %, DSR 0.85 — la gestion active apporte une vraie valeur.
- **DM non significatif** sur 948 jours pour AUCUNE comparaison : les différences sont économiquement notables (Sharpe varie de 0.88 à 1.71) mais ne franchissent pas le seuil de significativité statistique (p > 0.27). Limite intrinsèque d'un échantillon de ~3.8 ans.

→ **Recommandation mémoire : CNN-LSTM + HMM-gate** comme modèle de production. Le risk-gate seul (étape 7) reste utile comme défense additionnelle si on veut limiter l'exposition macro.